In [1]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

In [2]:
from generators.toy_problem_to_json import toy_problem_to_json # type: ignore[reportMissingImports]
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp_return_x # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [3]:
# Number of copies of the network in the toy system
N=10

# Whether the production and flow variables have upper bounds
F_UPPER_BOUNDS=True

# Whether the demand constraint is an equality or inequality
DEMAND_INEQUALITY=True

# Capacity / demand parameters
# GAMMA=1.0
GAMMA=1.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=0.5

LIN_APPROX = 2

# Whether to do an inner or outer approximation, THIS SHOULD STAY TRUE HERE
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [4]:
toy_problem_to_json(N, F_UPPER_BOUNDS, DEMAND_INEQUALITY, GAMMA, LAMBD,
                        EPS, DELTA, LP, LF, UP)
quad, q_time, q_x = solve_qcp_return_x(f_name="toy.json")
print(f"Exact objective value: {quad}")
print(f"Exact solution: {q_x}")

Set parameter Username
Set parameter LicenseID to value 2823041
Academic license - for non-commercial use only - expires 2027-05-16
Exact objective value: 11.100001521667886
Exact solution: [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.25071603371762596, 0.250657509264766, 0.25060248991752776, 0.25054176263952915, 0.250501431084549, 0.25047194259021127, 0.2504496147764327, 0.25043092308293663, 0.25041155659487635, 0.2503809162458384, 2.1995358397894396e-08, 1.8951755018295655e-08, 2.1768215587211122e-08, 2.192623321830308e-08, 2.194478172168556e-08, 2.195292829889449e-08, 2.1977684730571547e-08, 2.2020655187113113e-08, 2.2086391940627354e-08, 2.2170793402492293e-08, 0.4992814621794046, 0.005720533493733582, 0.5992867548996087, 0.5007132471042052, 0.4993412210733665, 0.0018739282811618418, 0.599344059326004, 0.5006559426485416, 0.004288748568440087, 0.49939698295620033, 0.001095610

In [5]:
quadratic_lin_approx_no_surrogate(1, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="toy.json", endpoints=False)
val, time, x = solve_lp_return_x()
print(x)

# print(condition_number_nes_basic())
print(f"val: {val}")
print(f"Error: {val - quad}")

[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.3750000000000001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.3749999999999999, 0.3321067811865477, 0.3928932188134525, 0.7071067811865476, 0.3749999999999999, 0.3321067811865477, 0.3928932188134525, 0.7071067811865476, 0.0, 0.3749999999999999, 0.3321067811865477, 0.3928932188134525, 0.7071067811865476, 0.0, 0.3749999999999999, 0.3321067811865477, 0.3928932188134525, 0.7071067811865476, 0.0, 0.3749999999999999, 0.3321067811865477, 0.3928932188134525, 0.7071067811865476, 0.0, 0.3749999999999999, 0.35355339059327356, 0.3928932188134525, 0.7071067811865476, 0.0, 0.3749999999999999, 0.35355339059327356, 0.3928932188134525, 0.7071067811865476, 0.0214466094

In [6]:
# Define a set of functions for each


quads = [[val] for val in x[(5 * N):]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
                refinement_points = len(qs)
                uniform = np.linspace(lower, upper, num - refinement_points)
                return np.append(uniform, qs)
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="toy.json", endpoints=False)

    val, time, x = solve_lp_return_x()

    # print(f"x: {x}")
    print(f"val: {val}")

    for j, qs in enumerate(quads):
        qs.append(x[(5 * N) + j])

    # print(quads)

print(x)
print(f"Error: {val - quad}")

Iteration 1
val: 11.06966991411009
Iteration 2
val: 11.08057347163526
Iteration 3
val: 11.09913269665953
Iteration 4
val: 11.09990625724199
Iteration 5
val: 11.099999248367567
[0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.24999922467546562, 0.24999962421837446, 0.24999962421837446, 0.24999962421837446, 0.2499996242183745, 0.24999962421837446, 0.24999962421837452, 0.24999962421837452, 0.2499996242183745, 0.24999962421837446, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5000007753245344, 0.0, 0.6000000230694338, 0.49999997693056636, 0.5000003757816256, 0.0, 0.5999996242183746, 0.5000003757816255, 7.983939679827558e-07, 0.5000003757816256, -7.983939682154606e-07, 0.5999996242183746, 0.5000003757816255, 7.983939681044383e-07, 0.5000003757816254, 0.0, 0.5999996242183746, 0.5000003757816255, 0.0, 0.5000003757816255, 0.0, 0.5999996242183746, 0.5000003757816255, 0.0, 0.500000375781